# Raw Exploratory Data Analysis (EDA): Configurable Diagnostics

This notebook performs exploratory data analysis on the raw MATLAB source file
containing 31 days of second-level (1 hertz or Hz) electrical load measurements for a
single commercial customer. It validates structural integrity, characterizes
the load distribution, identifies dominant periodicities, and assesses
day-class signal strength before bronze ingestion.

**Pipeline position**: First analytical step after raw data receipt. Precedes
bronze ingestion (`scripts/000_raw_to_bronze.py`).

**Input**: `data/000_raw/P_data.mat` -- a MATLAB binary containing three
arrays: `P_data` (86,400 rows x 31 columns of power load in watts),
`day_data` (calendar dates), and `day_class` (business-day labels: full, half,
none).

**Key outputs**:
- Per-day summary statistics and NaN audit
- Load distribution with adaptive outlier detection
- Power spectral density identifying daily and sub-daily cycles
- Day-class separation analysis (supports hypothesis H1)
- Day-class transition matrix
- Interactive drill-down timeline
- Data quality scorecard (6 pass/fail checks)

**Configuration**: All visualization and analysis parameters are governed by
[`config/eda.toml`](../config/eda.toml) and
[`config/pipeline.toml`](../config/pipeline.toml), loaded at runtime via
[`scripts/config.py`](../scripts/config.py). Environment variables
(`ELF_NB_AUTO_BINS`, `ELF_NB_AUTO_OUTLIER`) allow overrides without editing
the notebook.

**Related references**:
- Architecture: [`docs/001_architecture/001_raw/raw.md`](../docs/001_architecture/001_raw/raw.md)
- Pipeline: [`docs/002_pipeline/pipeline.md`](../docs/002_pipeline/pipeline.md)
- Hypotheses: [`docs/003_modeling/hypothesis.md`](../docs/003_modeling/hypothesis.md)

In [1]:
from pathlib import Path
import os
import sys
from typing import Literal, cast

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts' / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing scripts/config.py')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import PATHS, EDA_CONFIG
from scripts.utils import optimal_bin_count, adaptive_outlier_threshold

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # pyright: ignore[reportMissingModuleSource]
import plotly.express as px
from scipy import signal, stats
from scipy.io import loadmat

## Notebook Configuration

Configuration sources for this notebook:
- [`config/eda.toml`](../config/eda.toml)
- [`config/pipeline.toml`](../config/pipeline.toml)
- Runtime API: [`scripts/config.py`](../scripts/config.py)

Update values in those files first, then use the code cell below for notebook-scoped overrides.


In [2]:
# === NOTEBOOK CONFIGURATION ===
# Adjust parameters below to control analysis behavior.
# Defaults are imported from scripts.config EDA_CONFIG.
# To restore defaults, delete overrides and re-run this cell.

def _env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

RAW_PATH = PATHS['raw_mat']
FIGURE_SIZE = EDA_CONFIG['figure_size']
FIGURE_SIZE_WIDE = EDA_CONFIG['figure_size_wide']
DAY_CLASS_COLORS = EDA_CONFIG['day_class_colors']
SEABORN_STYLE = cast(Literal['white', 'dark', 'whitegrid', 'darkgrid', 'ticks'], str(EDA_CONFIG['seaborn_style']))

HISTOGRAM_BINS = EDA_CONFIG['histogram_bins']
ZSCORE_THRESHOLD = EDA_CONFIG['zscore_threshold']
PERCENTILES = EDA_CONFIG['percentiles']
LEGEND_MAX_LABELS = EDA_CONFIG['legend_max_labels']
LOAD_MIN_WATTS = EDA_CONFIG['physical_load_min_watts']
LOAD_MAX_WATTS = EDA_CONFIG['physical_load_max_watts']
AUTO_BINS = _env_bool('ELF_NB_AUTO_BINS', True)
AUTO_OUTLIER = _env_bool('ELF_NB_AUTO_OUTLIER', True)

sns.set_theme(style=SEABORN_STYLE)
plt.rcParams['figure.figsize'] = FIGURE_SIZE


## Load and Validate Raw Inputs

The raw layer contains a single MATLAB `.mat` file with three arrays: `P_data`
(86,400 rows x 31 columns of second-level power load), `day_data` (calendar
dates), and `day_class` (business-day labels). This cell loads the file,
parses all three arrays, and validates structural integrity before any analysis
begins.

In [3]:
# --- Load MATLAB file and extract the three primary arrays ---
raw = loadmat(RAW_PATH)
p_data = np.asarray(raw['P_data'], dtype=float)

def extract_values(arr):
    """Unwrap a MATLAB cell array into a flat Python list.

    MATLAB cell arrays are stored as nested numpy object arrays. This function
    handles byte-string decoding, single-element unwrapping, and string-array
    joining to produce clean Python values for downstream use.
    """
    vals = []
    for item in np.asarray(arr).squeeze():
        v = item
        # Recursively unwrap single-element numpy arrays
        while isinstance(v, np.ndarray) and v.size == 1:
            v = v.item()
        # Decode byte strings (common in MATLAB v5 .mat files)
        if isinstance(v, bytes):
            v = v.decode('utf-8')
        # Join character arrays into a single string
        if isinstance(v, np.ndarray) and v.dtype.kind in {'U', 'S'}:
            v = ''.join(v.tolist())
        vals.append(v)
    return vals

# --- Parse dates and day-class labels ---
dates = pd.to_datetime(extract_values(raw['day_data']))
day_classes = (
    pd.Series(extract_values(raw['day_class']))
    .astype(str)
    .str.lower()
    .str.strip()
)

# --- Validate structural integrity ---
assert p_data.shape[1] == len(dates) == len(day_classes), (
    f'Shape mismatch: P_data has {p_data.shape[1]} columns, '
    f'but dates={len(dates)} and day_classes={len(day_classes)}'
)
assert set(day_classes.unique()) <= {'full', 'half', 'none'}, (
    f'Unexpected day_class values: {set(day_classes.unique())}'
)

print('shape:', p_data.shape)
print('date range:', dates.min(), '->', dates.max())

In [4]:
# --- Raw Data Health Check Card ---
import numpy as np, pandas as pd

checks = []

# Shape check
shape_ok = p_data.shape == (86400, 31)
checks.append(("Shape (86400, 31)", str(p_data.shape), "PASS" if shape_ok else "FAIL"))

# Arrays present
arrays_ok = all(k in raw for k in ['P_data', 'day_data', 'day_class'])
checks.append(("Arrays present", "P_data, day_data, day_class", "PASS" if arrays_ok else "FAIL"))

# NaN rate
nan_pct = float(np.isnan(p_data).mean() * 100)
nan_ok = nan_pct < 2.0
checks.append(("NaN rate < 2%", f"{nan_pct:.2f}%", "PASS" if nan_ok else "FAIL"))

# Day classes valid
dc_set = set(day_classes.unique())
dc_ok = dc_set <= {'full', 'half', 'none'}
checks.append(("Day classes valid", ", ".join(sorted(dc_set)), "PASS" if dc_ok else "FAIL"))

# Date range
dr_str = f"{dates.min().strftime('%b %d')} - {dates.max().strftime('%b %d, %Y')}"
dr_ok = len(dates) == 31
checks.append(("Date range", dr_str, "PASS" if dr_ok else "FAIL"))

# Physical range
non_nan_vals = p_data[np.isfinite(p_data)]
phys_ok = bool(((non_nan_vals >= LOAD_MIN_WATTS) & (non_nan_vals <= LOAD_MAX_WATTS)).all())
checks.append(("Physical range", f"[{LOAD_MIN_WATTS}, {LOAD_MAX_WATTS}]", "PASS" if phys_ok else "FAIL"))

# Print card
n_pass = sum(1 for _, _, s in checks if s == "PASS")
overall = "PASS" if n_pass == len(checks) else "FAIL"

print("RAW DATA HEALTH CHECK")
print("=" * 55)
for name, metric, status in checks:
    print(f"  {name:<25s}  {metric:<20s}  {status}")
print("=" * 55)
print(f"  Overall: {n_pass}/{len(checks)} checks passed    {overall}")
print(f"  Next step: Data is ready for bronze ingestion (scripts/000_raw_to_bronze.py)")


## Summary Statistics, NaN Patterns, and Load Distribution

Three diagnostics establish baseline data quality:

1. **Per-day summary table** (tabular summary)
   - **Purpose**: Reports mean load, standard deviation, NaN count, and NaN
     percentage for each of the 31 calendar days alongside the day class.
   - **Look for**: Days with unusually high NaN rates or load statistics that
     diverge sharply from same-class peers, which may indicate data collection
     issues or anomalous operating conditions.

2. **NaN heatmap** (heatmap, date x hour)
   - **Purpose**: Visualizes missingness density across calendar days and hours
     of the day.
   - **Look for**: Clustered blocks of NaN (full hours missing) suggest sensor
     outages during specific time windows. Scattered NaN suggests intermittent
     read failures. Concentrated missingness in specific hours indicates
     time-dependent reliability issues.

3. **Load distribution histogram** (histogram)
   - **Purpose**: Shows the overall shape of the load signal using adaptive
     (Freedman-Diaconis) or fixed bin counts.
   - **Look for**: Right skew driven by business-hour peaks, multimodality
     indicating distinct operating regimes, or gaps in the distribution that
     may indicate sensor quantization. The reported outlier rate quantifies
     the fraction of readings outside adaptive Interquartile Range (IQR) bounds.

In [5]:
# --- Per-day summary table ---
# Compute column-wise (per-day) statistics from the raw (86400, 31) matrix.
daily = pd.DataFrame({
    'date': dates,
    'day_class': day_classes,
    'mean': np.nanmean(p_data, axis=0),
    'std': np.nanstd(p_data, axis=0),
})
daily['nan_count'] = np.isnan(p_data).sum(axis=0)
daily['nan_pct'] = daily['nan_count'] / p_data.shape[0] * 100

# Flatten to 1-D for distribution and percentile analysis (exclude NaN/Inf).
flat = p_data[np.isfinite(p_data)]
q = pd.DataFrame({
    'percentile': PERCENTILES,
    'value': np.quantile(flat, PERCENTILES),
})

display(daily.head(10))
display(q)

# --- NaN heatmap ---
# Reshape the boolean NaN mask into (24 hours, 3600 seconds/hour, days),
# then average across seconds to get the NaN rate per hour per day.
nan_hourly = np.isnan(p_data).reshape(24, 3600, p_data.shape[1]).mean(axis=1).T

plt.figure(figsize=FIGURE_SIZE_WIDE)
sns.heatmap(
    pd.DataFrame(nan_hourly, index=dates.strftime('%Y-%m-%d')),
    cmap='rocket_r',
    vmin=0, vmax=1,
    cbar_kws={'label': 'NaN Rate', 'shrink': 0.8},
    linewidths=0.3, linecolor='#f0f0f0',
)
plt.xlabel('Hour of Day')
plt.ylabel('Date')
plt.title('NaN Rate by Day and Hour')
plt.tight_layout()
plt.show()

# --- Load distribution histogram ---
# Bin count is either data-adaptive (Freedman-Diaconis) or the fixed default.
bins = optimal_bin_count(flat) if AUTO_BINS else HISTOGRAM_BINS

# Outlier detection: adaptive IQR per day class or fixed z-score threshold.
if AUTO_OUTLIER:
    bounds = adaptive_outlier_threshold(flat, method='iqr')
    outlier_mask = (flat < float(bounds['lower'])) | (flat > float(bounds['upper']))
else:
    mu, sigma = float(flat.mean()), float(flat.std(ddof=0))
    outlier_mask = np.abs((flat - mu) / sigma) > ZSCORE_THRESHOLD

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.hist(flat, bins=bins, color='#1f77b4', edgecolor='white', linewidth=0.3, alpha=0.85)
ax.axvline(float(np.median(flat)), color='#e45756', linestyle='--', linewidth=1.2, label=f'Median: {float(np.median(flat)):,.0f} W')
ax.axvline(float(np.mean(flat)), color='#f58518', linestyle=':', linewidth=1.2, label=f'Mean: {float(np.mean(flat)):,.0f} W')
ax.set_xlabel('Load (watts)')
ax.set_ylabel('Frequency')
ax.set_title('Raw Load Distribution')
ax.legend(frameon=True, fancybox=False, edgecolor='#cccccc')
ax.grid(axis='y', alpha=0.3, linestyle='--')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print(f'Bins: {bins} | Outlier rate: {float(outlier_mask.mean() * 100):.2f}%')

## Load Profiles, Normality, Spectral Analysis, and Day-Class Behavior

Five analyses characterize the temporal and distributional properties of the
raw signal:

1. **Per-day minute-level profiles** (line overlay)
   - **Purpose**: Overlays all 31 days at one-minute resolution on a shared
     time axis to reveal day-to-day consistency.
   - **Look for**: Days that deviate sharply from their class peers (potential
     anomalies or holidays). Tight clustering within a class confirms
     repeatable consumption patterns.

2. **Quantile-Quantile (QQ) plot** (scatter)
   - **Purpose**: Compares empirical load quantiles against a theoretical
     normal distribution to assess distributional fit.
   - **Look for**: Departures from the diagonal at the tails indicate
     non-normality (heavy tails, skew). This informs whether downstream
     models that assume normally distributed residuals will need transforms.

3. **Power Spectral Density (PSD)** (log-scale line, period domain)
   - **Purpose**: Decomposes the signal into frequency components to identify
     dominant periodicities. Interpolation fills NaN gaps before spectral
     estimation, which may slightly attenuate high-frequency content near gaps.
   - **Look for**: A strong peak at 1,440 minutes (24 hours) confirms daily
     cycling. A secondary peak at 720 minutes (12 hours) indicates a
     business-peak harmonic. Unexpected peaks may reveal equipment cycling or
     periodic scheduling patterns.

4. **Day-class stratified box plots** (grouped box plot)
   - **Purpose**: Compares the distribution of daily mean load across the three
     day types to evaluate day-class signal strength.
   - **Look for**: Clear separation between classes supports `workday` as a
     predictor (hypothesis H1). Overlapping boxes would indicate weak
     day-class signal.

5. **Day-class transition matrix** (annotated heatmap)
   - **Purpose**: Shows the empirical probability of moving from one day type
     to the next across consecutive calendar days.
   - **Look for**: A diagonal-heavy matrix indicates stable scheduling.
     Off-diagonal mass suggests irregular patterns such as holidays or partial
     weeks that the model may encounter during inference.

In [6]:
# --- Per-day 1-minute profiles ---
# Reshape the (86400, 31) matrix into (1440 minutes, 60 seconds, 31 days)
# and average across the 60-second axis to get minute-level profiles.
minute_profiles = np.nanmean(p_data.reshape(1440, 60, p_data.shape[1]), axis=1)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
# Plot each day color-coded by day class for visual grouping.
legend_handles = {}
for i in range(minute_profiles.shape[1]):
    cls = day_classes.iloc[i]
    color = DAY_CLASS_COLORS.get(cls, 'gray')
    ax.plot(minute_profiles[:, i], alpha=0.35, linewidth=0.8, color=color)
    if cls not in legend_handles:
        legend_handles[cls] = ax.plot([], [], color=color, linewidth=2, label=cls)[0]
ax.legend(handles=list(legend_handles.values()), loc='upper right', frameon=True, fancybox=False, edgecolor='#cccccc')
ax.set_xlabel('Minute of Day')
ax.set_ylabel('Load (watts)')
ax.set_title('Per-Day Load Profiles (1-Minute Resolution)')
ax.grid(axis='both', alpha=0.2, linestyle='--')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# --- QQ plot ---
# Subsample every 50th value for computational efficiency. The subsampling
# preserves the distributional shape while keeping the plot responsive.
def to_float_scalar(value: object, field_name: str) -> float:
    # Convert scalar-like outputs from SciPy into strict Python floats.
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    arr = np.asarray(value)
    if arr.ndim != 0:
        raise TypeError(f"{field_name} must be scalar-like; got shape={arr.shape}.")

    scalar = arr.item()
    if not isinstance(scalar, (int, float, np.integer, np.floating)):
        raise TypeError(f"{field_name} must be numeric; got type={type(scalar).__name__}.")
    return float(scalar)


fig, ax = plt.subplots(figsize=FIGURE_SIZE)
(osm, osr), fit_params = stats.probplot(flat[::50], dist='norm')
slope_f = to_float_scalar(fit_params[0], 'slope')
intercept_f = to_float_scalar(fit_params[1], 'intercept')
osm_arr = np.asarray(osm, dtype=float)
osr_arr = np.asarray(osr, dtype=float)
ax.scatter(osm_arr, osr_arr, s=8, alpha=0.5, color='#4c78a8', edgecolors='none', label='Sample quantiles')
ax.plot(osm_arr, slope_f * osm_arr + intercept_f, color='#e45756', linewidth=1.5, label='Reference line')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles (watts)')
ax.set_title('QQ Plot: Load vs. Normal Distribution (Sampled)')
ax.legend(frameon=True, fancybox=False, edgecolor='#cccccc')
ax.grid(alpha=0.2, linestyle='--')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# --- Power spectral density ---
# Interpolation fills NaN gaps in the minute-level signal before frequency-
# domain analysis. This is acceptable for identifying dominant periodicities
# but may slightly attenuate high-frequency content near gaps.
minute_series = (
    pd.Series(minute_profiles.T.reshape(-1))
    .interpolate(limit_direction='both')
    .to_numpy(float)
)
# Welch's method: fs=1.0 means 1 sample per minute; nperseg controls the
# frequency resolution vs. variance tradeoff.
freq, power = signal.welch(minute_series, fs=1.0, nperseg=min(2048, len(minute_series)))
period = np.where(freq > 0, 1.0 / freq, np.nan)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
ax.semilogy(period, power, color='#4c78a8', linewidth=1.2)
# Annotate dominant periodicities: 24-hour daily cycle and 12-hour harmonic.
ax.axvline(1440, color='#e45756', linestyle='--', linewidth=1, alpha=0.7, label='24h (1440 min)')
ax.axvline(720, color='#f58518', linestyle='--', linewidth=1, alpha=0.7, label='12h (720 min)')
ax.set_xlim(0, 24 * 60)
ax.set_xlabel('Period (minutes)')
ax.set_ylabel('Power Spectral Density')
ax.set_title('Power Spectral Density of Minute-Level Load')
ax.legend(frameon=True, fancybox=False, edgecolor='#cccccc')
ax.grid(axis='both', alpha=0.2, linestyle='--')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# --- Day-class stratified box plots ---
class_df = pd.DataFrame({
    'day_class': day_classes,
    'daily_mean': np.nanmean(p_data, axis=0),
})

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
sns.boxplot(
    data=class_df,
    x='day_class',
    y='daily_mean',
    order=['none', 'half', 'full'],
    palette=DAY_CLASS_COLORS,
    width=0.5,
    flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5},
    ax=ax,
)
# Overlay individual day points for context at n=31.
sns.stripplot(
    data=class_df,
    x='day_class',
    y='daily_mean',
    order=['none', 'half', 'full'],
    palette=DAY_CLASS_COLORS,
    size=5, alpha=0.6, jitter=0.15,
    ax=ax,
)
ax.set_xlabel('Day Class')
ax.set_ylabel('Daily Mean Load (watts)')
ax.set_title('Daily Mean Load by Business-Day Class')
ax.grid(axis='y', alpha=0.2, linestyle='--')
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# --- Day-class transition matrix ---
# Row-normalized crosstab: each row sums to 1.0, showing the probability of
# transitioning from the previous day's class to the next day's class.
transition = pd.crosstab(
    day_classes.shift(1), day_classes, normalize='index',
).fillna(0.0)

fig, ax = plt.subplots(figsize=FIGURE_SIZE)
sns.heatmap(
    transition, annot=True, fmt='.2f', cmap='Blues', cbar=False,
    linewidths=0.5, linecolor='white',
    annot_kws={'fontsize': 12, 'fontweight': 'bold'},
    ax=ax,
)
ax.set_xlabel('Next Day Class')
ax.set_ylabel('Previous Day Class')
ax.set_title('Day-Class Transition Probabilities')
plt.tight_layout()
plt.show()


## Interactive Drill-Down and Data Quality Scorecard

1. **Interactive load timeline** (Plotly line chart, zoomable)
   - **Purpose**: Provides a pan-and-zoom view of the full 31-day load signal
     at one-minute resolution, color-coded by day class, with hover tooltips
     for precise value inspection.
   - **Look for**: Anomalous periods identified in earlier plots can be
     isolated by zooming. Use the legend to toggle individual day classes on
     or off to compare consumption patterns.

2. **Data quality scorecard** (pass/fail table)
   - **Purpose**: Aggregates six structural checks into a single readiness
     assessment for the raw layer.
   - **Look for**: All checks should pass before proceeding to bronze
     ingestion. The outlier rate check may legitimately fail when using
     IQR-based detection on a heavily skewed commercial load distribution --
     this flags the long upper tail of high-load business hours rather than
     true sensor anomalies.

## Data Lineage & Quality Gate

**Input:** Raw MATLAB file (`data/000_raw/P_data.mat`)
**Producer:** Original instrumentation / measurement system
**Consumer:** `scripts/000_raw_to_bronze.py`

All scorecard checks below must pass (or be documented as expected caveats)
before running the consumer script. Critical failures block advancement;
expected caveats are labeled.


In [7]:
# --- Interactive drill-down timeline (Plotly) ---
# Build a minute-level DataFrame for all 31 days with day-class color coding.
# np.repeat tiles each date/class value across 1,440 minutes; np.tile repeats
# the minute index across all days.
date_values = dates.to_numpy(dtype='datetime64[ns]')
day_class_values = day_classes.astype(str).to_numpy()

timeline = pd.DataFrame({
    'date': np.repeat(date_values, 1440),
    'minute': np.tile(np.arange(1440), len(dates)),
    'avg_load': minute_profiles.T.reshape(-1),
    'day_class': np.repeat(day_class_values, 1440),
})
timeline['timestamp'] = timeline['date'] + pd.to_timedelta(timeline['minute'], unit='m')

fig = px.line(
    timeline,
    x='timestamp',
    y='avg_load',
    color='day_class',
    color_discrete_map=DAY_CLASS_COLORS,
    labels={'avg_load': 'Load (watts)', 'timestamp': 'Timestamp', 'day_class': 'Day Class'},
    title='Interactive Raw Load (1-Minute Resolution)',
)
fig.update_traces(opacity=0.7)
fig.update_layout(
    xaxis_title='Timestamp',
    yaxis_title='Load (watts)',
    legend_title_text='Day Class',
    hovermode='x unified',
    template='plotly_white',
)
fig.show()

# --- Data quality scorecard ---
# Each check evaluates one structural property of the raw payload. A passing
# scorecard confirms the file is suitable for bronze ingestion.
non_nan = p_data[np.isfinite(p_data)]
physical_ok = bool(((non_nan >= LOAD_MIN_WATTS) & (non_nan <= LOAD_MAX_WATTS)).all())

score = pd.DataFrame([
    {'check': 'shape_(86400,31)',    'metric': str(p_data.shape),                           'pass': p_data.shape == (86400, 31)},
    {'check': 'nan_rate_lt_2pct',    'metric': f'{float(np.isnan(p_data).mean() * 100):.2f}%', 'pass': float(np.isnan(p_data).mean() * 100) < 2},
    {'check': 'date_uniqueness_31',  'metric': int(pd.Index(dates).nunique()),              'pass': int(pd.Index(dates).nunique()) == 31},
    {'check': 'day_class_valid',     'metric': ','.join(sorted(day_classes.unique())),       'pass': set(day_classes.unique()) <= {'full', 'half', 'none'}},
    {'check': 'physical_range',      'metric': f'[{LOAD_MIN_WATTS}, {LOAD_MAX_WATTS}]',     'pass': physical_ok},
    {'check': 'outlier_rate_lt_1pct','metric': f'{float(outlier_mask.mean() * 100):.2f}%',   'pass': float(outlier_mask.mean() * 100) < 1},
])
display(score)

## Layer Health Summary

**Status: PASS** — All structural checks passed. Raw data is ready for bronze
ingestion.

**Caveats:**
- **outlier_rate_lt_1pct = False (12.17%)**: This is **EXPECTED** for
  right-skewed commercial load data. IQR-based detection flags the natural
  upper tail of business-hour peaks, not measurement errors. Commercial
  facilities exhibit sharp daytime-to-nighttime load ratios (4:1 or higher),
  which concentrates the upper quartile and inflates IQR-based outlier counts.
  This does not block bronze ingestion.

**Next step:** Run `scripts/000_raw_to_bronze.py` to advance to bronze layer.
If any critical check above showed FAIL (not an expected caveat), investigate
before proceeding.


## Key Findings

- **Data integrity confirmed.** The raw payload has the expected shape
  (86,400 x 31), all 31 dates are unique, and day-class labels are valid.
  NaN rate is 0.54%, well below the 2% warning threshold.
- **Load distribution is right-skewed.** The QQ plot and histogram show a long
  upper tail driven by daytime business-hour peaks. This skew makes
  IQR-based outlier detection more appropriate than symmetric z-score methods
  for this dataset.
- **Strong daily periodicity.** The PSD confirms a dominant 24-hour cycle with
  a secondary 12-hour harmonic, consistent with a commercial facility that
  ramps up in the morning and powers down in the evening.
- **Day-class separation is clear.** Full working days have substantially
  higher mean load than half or non-working days, supporting the use of
  `workday` as a predictor feature (hypothesis H1).
- **Transition patterns are regular.** The transition matrix reveals
  predictable scheduling sequences that the model may be able to exploit.

## Glossary

| Term | Definition | Context |
|------|------------|---------|
| `day_class` | Customer-provided label classifying each calendar day as `full`, `half`, or `none` based on operational schedule. | Raw metadata column; drives the `workday` predictor downstream. |
| `day_data` | Array of calendar dates in the MATLAB payload, one per column of `P_data`. | Raw layer; used to construct timestamps during bronze ingestion. |
| `full` / `half` / `none` | Business-day classifications indicating full working day, partial working day, or non-working day respectively. | Applied by the customer; encoded as ternary `workday` in silver. |
| Histogram | Bar chart showing the frequency distribution of a continuous variable across bins. | Used here to visualize the overall load distribution. |
| IQR | Interquartile range (Q3 minus Q1). A robust measure of spread that is less sensitive to outliers than standard deviation. | Used in adaptive outlier detection when `AUTO_OUTLIER` is enabled. |
| Load | Instantaneous power consumption measured in watts at one-second granularity. | The core signal throughout the pipeline. |
| MATLAB (`.mat`) | Binary file format produced by MathWorks MATLAB software. The raw data source for this project. | Read via `scipy.io.loadmat` during bronze ingestion. |
| NaN | "Not a Number" marker representing a missing or invalid measurement. | 14,576 NaN values exist in the raw data (0.54% of total). |
| NaN rate | The percentage of values that are NaN within a given scope (column, day, or overall dataset). | Key data-quality metric; monitored at every pipeline layer. |
| Outlier | An observation that falls outside expected bounds, detected via IQR fencing or z-score thresholds. | Flagged for awareness; not automatically removed at the raw layer. |
| `P_data` | The primary data matrix in the MATLAB payload, shaped (86,400 x d) where d is the number of days. | Each row is one second of the day; each column is one calendar day. |
| Percentile | A value below which a given percentage of observations fall (e.g., the 95th percentile). | Used in summary statistics and outlier boundary estimation. |
| PSD | Power spectral density -- a frequency-domain representation showing how signal power is distributed across frequencies. | Identifies dominant periodicities such as the 24-hour daily cycle. |
| QQ plot | Quantile-quantile plot comparing the empirical distribution of a sample against a theoretical reference distribution. | Departures from the diagonal indicate non-normality (heavy tails, skew). |
| Raw layer | The first layer in the medallion architecture containing untouched customer data. The pipeline never modifies the raw file. | Documented in `docs/001_architecture/001_raw/raw.md`. |
| Second-level data | Data sampled at one measurement per second (1 Hz), producing 86,400 readings per day. | The native granularity of the raw source data. |
| Standard deviation | A measure of the average distance of observations from the mean. | Used in z-score computation and summary statistics. |
| Transition matrix | A table showing the probability of transitioning from one state to another between consecutive time steps. | Here, shows day-class-to-day-class transition probabilities. |
| Z-score | The number of standard deviations an observation lies from the mean. Values beyond a threshold (default 3.0) are flagged as outliers. | Used when `AUTO_OUTLIER` is disabled as a fixed-threshold alternative. |